In [ ]:
# git clone https://github.com/AgentschapPlantentuinMeise/MeiseCSI
# cd MeiseCSI
# python -m venv venv
# . venv/bin/activate
# pip install ipython pandas jupyter
# jupyter lab
import pandas as pd
import numpy as np

In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning, message="Mean of empty slice")

def select_wavelengths(
    df: pd.DataFrame,
    min_corr_coeff: float =0.995,
    max_conc_rsd: float =5,
    max_bias: float =10,
    use_q_flag: bool =True,
    use_rse: bool =True
):
    """
    Production‑ready wavelength selection for Agilent 5800 ICP‑OES.
    Uses:
      - Element Symbol
      - Wavelength
      - Radial flag
    """

    # ---------------------------------------------------------
    # 0. SPLIT ELEMENT COLUMN INTO SYMBOL + WAVELENGTH
    # ---------------------------------------------------------
    # Example: "Ag 224.641_R"
    df[["Element Symbol", "Wavelength"]] = (
        df["Element"]
        .astype(str)
        .str.split(n=1, expand=True)
    )

    # ---------------------------------------------------------
    # 1. RADIAL FLAG + CLEAN WAVELENGTH
    # ---------------------------------------------------------
    df["Radial"] = df["Element Label"].str.endswith("_R")

    # ---------------------------------------------------------
    # 2. ULTRA‑ROBUST NUMERIC CLEANING
    # ---------------------------------------------------------
    def clean_numeric(series):
        cleaned = (
            series.astype(str)
                  .str.replace(",", ".", regex=False)
                  .str.replace(r"[A-Za-z#]", "", regex=True)
                  .str.replace("<|>|—|–|-", "", regex=True)
                  .str.strip()
        )

        # Replace empty strings with NaN without warnings
        cleaned = cleaned.mask(cleaned == "", np.nan)

        return pd.to_numeric(cleaned, errors="coerce")


    numeric_cols = [
        "Concentration",
        "Concentration % RSD",
        "Intensity % RSD",
        "Correlation coefficient",
        "%RSE",
        "%RSE limit",
    ]

    for col in numeric_cols:
        df[col] = clean_numeric(df[col])

    # ---------------------------------------------------------
    # 3. BASIC FLAGS & QC DETECTION
    # ---------------------------------------------------------
    df["_is_crm"] = df["Type"] == "QC"
    df["_has_q_flag"] = df["Flags"].astype(str).str.contains("Q", na=False)

    df_crm = df[df["_is_crm"]]
    df_unk = df[~df["_is_crm"]]

    # ---------------------------------------------------------
    # 4. CRM BIAS CALCULATION
    # ---------------------------------------------------------
    crm_bias = None
    if not df_crm.empty:
        crm_bias = (
            df_crm.groupby(["Element Symbol", "Wavelength", "Radial"])["Concentration"]
                  .apply(lambda x: x.mean())
                  .rename("crm_mean")
        )

    # ---------------------------------------------------------
    # 5. WAVELENGTH SCORING
    # ---------------------------------------------------------
    def score_wavelength(group):
        g_unk = group[~group["_is_crm"]]
        if g_unk.empty:
            g_unk = group

        conc_rsd = g_unk["Concentration % RSD"].median()
        intensity_rsd = g_unk["Intensity % RSD"].median()
        corr_coeff = group["Correlation coefficient"].max()
        rse = group["%RSE"].median()
        rse_limit = group["%RSE limit"].median()
        q_flag = group["_has_q_flag"].any()

        key = group.name

        bias = crm_bias.loc[key] if crm_bias is not None and key in crm_bias.index else np.nan

        all_nan_conc = group["Concentration"].isna().all()

        reject_reasons = []
        if all_nan_conc:
            reject_reasons.append("no numeric data")

        if use_q_flag and q_flag:
            reject_reasons.append("Q flag")

        if corr_coeff < min_corr_coeff:
            reject_reasons.append("low corr coeff")

        if use_rse and rse > rse_limit:
            reject_reasons.append("RSE > limit")
        
        if conc_rsd > max_conc_rsd:
            reject_reasons.append("high %RSD")

        if use_q_flag and not np.isnan(bias) and abs(bias) > max_bias:
            reject_reasons.append("high CRM bias")

        rejected = len(reject_reasons) > 0

        score = 0
        if not rejected:
            score += max(0, 10 - conc_rsd)
            score += (corr_coeff - 0.995) * 100
            if not np.isnan(bias):
                score += max(0, 10 - abs(bias))

        return pd.Series({
            "wavelength": key[1],
            "radial": key[2],
            "rejected": rejected,
            "reject_reasons": "; ".join(reject_reasons),
            "score": score,
            "conc_rsd": conc_rsd,
            "intensity_rsd": intensity_rsd,
            "corr_coeff": corr_coeff,
            "rse": rse,
            "rse_limit": rse_limit,
            "crm_bias": bias
        })

    wl_scores = (
        df.groupby(["Element Symbol", "Wavelength", "Radial"])
          .apply(score_wavelength, include_groups=False)
          .reset_index()
    )

    # ---------------------------------------------------------
    # 6. FINAL SELECTION PER SAMPLE
    # ---------------------------------------------------------
    results = []
    group_cols = ["Label", "Rack:Tube", "Element Symbol"]

    for key, g in df_unk.groupby(group_cols):
        element = key[2]

        g = g.merge(
            wl_scores[wl_scores["Element Symbol"] == element][
                ["Wavelength", "Radial", "rejected", "score"]
            ],
            on=["Wavelength", "Radial"],
            how="left"
        )

        valid = g[~g["rejected"]]
        if valid.empty:
            results.append({
                "Label": key[0],
                "Rack:Tube": key[1],
                "Element Symbol": element,
                "final_concentration": np.nan,
                "decision": "all_rejected",
                "used_wavelengths": ""
            })
            continue

        wl_conc = (
            valid.groupby(["Wavelength", "Radial"])["Concentration"]
                 .apply(lambda x: x.mean())
                 .rename("mean_conc")
                 .reset_index()
        )

        wl_conc = wl_conc.merge(
            wl_scores[wl_scores["Element Symbol"] == element][
                ["Wavelength", "Radial", "score"]
            ],
            on=["Wavelength", "Radial"],
            how="left"
        ).sort_values("score", ascending=False)

        top = wl_conc.iloc[0]

        similar = [top]
        for _, row in wl_conc.iloc[1:].iterrows():
            if np.isnan(top["mean_conc"]) or np.isnan(row["mean_conc"]) or top["mean_conc"] == 0:
                continue
            diff = abs(row["mean_conc"] - top["mean_conc"]) / top["mean_conc"] * 100
            if diff <= 10:
                similar.append(row)

        if len(similar) > 1:
            final = np.mean([r["mean_conc"] for r in similar])
            used = [f"{r['Wavelength']}{'_R' if r['Radial'] else ''}" for r in similar]
            decision = "average"
        else:
            final = top["mean_conc"]
            used = [f"{top['Wavelength']}{'_R' if top['Radial'] else ''}"]
            decision = "single"

        results.append({
            "Label": g["Label"].iloc[0],
            "Rack:Tube": g["Rack:Tube"].iloc[0],
            "Element Symbol": element,
            "final_concentration": final,
            "decision": decision,
            "used_wavelengths": ", ".join(used)
        })

    decisions_df = pd.DataFrame(results)
    final_df = (
        decisions_df
            .pivot_table( 
                index=["Label", "Rack:Tube"],
                columns="Element Symbol", 
                values="final_concentration" 
            )
    )
    # ---------------------------------------------------------
    # 7. QC SUMMARY TABLE
    # ---------------------------------------------------------
    qc_summary = wl_scores.copy()
    qc_summary["status"] = np.where(qc_summary["rejected"], "Rejected", "Accepted")

    return final_df, decisions_df, wl_scores, qc_summary


In [ ]:
# Recalibration and measuring LOD + LOQ
import numpy as np
import pandas as pd

def calibrate_and_compute_lod(df, wl_scores, min_std_count=10):
    """
    Adds calibration slope, intercept, R², LOD, and LOQ to wl_scores.
    Calibration is linear with intercept: I = m*C + b.
    LOD and LOQ are computed from blank intensity noise.
    """

    # ---------------------------------------------------------
    # 1. Identify valid calibration levels (expected concentrations)
    # ---------------------------------------------------------
    std_counts = (
        df[df["Type"] == "STD"]["Concentration"]
        .value_counts()
    )
    known_levels = std_counts[std_counts >= min_std_count].index.to_list()

    # ---------------------------------------------------------
    # 2. Helper: build calibration model
    # ---------------------------------------------------------
    def build_calibration(cal):
        conc = cal["Concentration"].astype(float)
        inten = cal["Intensity"].astype(float)

        # Fit linear regression with intercept
        m, b = np.polyfit(conc, inten, 1)

        # Compute R²
        pred = m * conc + b
        ss_res = np.sum((inten - pred)**2)
        ss_tot = np.sum((inten - np.mean(inten))**2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

        return m, b, r2

    # ---------------------------------------------------------
    # 3. Helper: compute LOD and LOQ from blank intensities
    # ---------------------------------------------------------
    def compute_lod_loq(blank_intensities, slope):
        if slope == 0 or np.isnan(slope) or len(blank_intensities) < 2:
            return np.nan, np.nan

        sd_blank = np.std(blank_intensities, ddof=1)
        lod = 3 * sd_blank / slope
        loq = 10 * sd_blank / slope
        return lod, loq

    # ---------------------------------------------------------
    # 4. Loop through each wavelength and compute calibration + LOD
    # ---------------------------------------------------------
    slopes = []
    intercepts = []
    r2s = []
    lods = []
    loqs = []

    for idx, row in wl_scores.iterrows():
        elem = row["Element Symbol"]
        wl = row["Wavelength Clean"]
        rad = row["Radial"]

        # Extract calibration standards for this wavelength
        cal = df[
            (df["Element Symbol"] == elem) &
            (df["Wavelength Clean"] == wl) &
            (df["Radial"] == rad) &
            (df["Type"] == "STD")
        ]

        # Keep only valid calibration levels
        cal = cal[cal["Concentration"].isin(known_levels)]

        # Extract blanks
        blanks = df[
            (df["Element Symbol"] == elem) &
            (df["Wavelength Clean"] == wl) &
            (df["Radial"] == rad) &
            (df["Type"] == "BLK")
        ]

        # Build calibration
        if len(cal) >= 2:
            m, b, r2 = build_calibration(cal)
        else:
            m, b, r2 = np.nan, np.nan, np.nan

        # Compute LOD/LOQ
        lod, loq = compute_lod_loq(blanks["Intensity"].dropna(), m)

        slopes.append(m)
        intercepts.append(b)
        r2s.append(r2)
        lods.append(lod)
        loqs.append(loq)

    # ---------------------------------------------------------
    # 5. Attach results to wl_scores
    # ---------------------------------------------------------
    wl_scores["slope"] = slopes
    wl_scores["intercept"] = intercepts
    wl_scores["r2"] = r2s
    wl_scores["LOD"] = lods
    wl_scores["LOQ"] = loqs

    return wl_scores


In [ ]:
def diagnostic_report(df, wl_scores, results_df):
    report = {}

    # 1. Elements in raw data
    raw_elements = set(df["Element Symbol"].unique())
    report["raw_elements"] = raw_elements

    # 2. Elements with at least one wavelength group
    wl_elements = set(wl_scores["Element Symbol"].unique())
    report["wl_elements"] = wl_elements

    # 3. Elements fully rejected at wavelength QC stage
    rejected_elements = set(
        wl_scores.groupby("Element Symbol")["rejected"].all()
        .loc[lambda s: s].index
    )
    report["rejected_elements"] = rejected_elements

    # 4. Elements with no unknown samples
    unk_elements = set(df.loc[~df["_is_crm"], "Element Symbol"].unique())
    no_unknowns = raw_elements - unk_elements
    report["no_unknowns"] = no_unknowns

    # 5. Elements that passed QC but still missing in final results
    final_elements = set(results_df["Element Symbol"].unique())
    passed_qc_missing = (wl_elements - rejected_elements) - final_elements
    report["passed_qc_missing"] = passed_qc_missing

    # 6. Elements in final results
    report["final_elements"] = final_elements

    # 7. Detailed per-element explanation
    details = {}
    for elem in sorted(raw_elements):
        details[elem] = {
            "in_raw_data": elem in raw_elements,
            "has_wavelengths": elem in wl_elements,
            "all_wavelengths_rejected": elem in rejected_elements,
            "has_unknowns": elem in unk_elements,
            "appears_in_final": elem in final_elements,
            "missing_reason": None
        }

        if elem not in wl_elements:
            details[elem]["missing_reason"] = "No wavelength groups found"
        elif elem in rejected_elements:
            details[elem]["missing_reason"] = "All wavelengths failed QC"
        elif elem not in unk_elements:
            details[elem]["missing_reason"] = "No unknown samples"
        elif elem not in final_elements:
            details[elem]["missing_reason"] = "No valid concentration after QC"
        else:
            details[elem]["missing_reason"] = "OK"

    report["details"] = details

    return report


In [ ]:
#data = pd.read_csv("data/icpoes/decaf_coffees.csv", skiprows=6)
data = pd.read_csv("data/icpoes/wfid_samples_3_redux.csv", skiprows=6, low_memory=False)
data

In [ ]:
data.Description = data.Label.apply(lambda x: ' '.join(x.split()[:-1]))

In [ ]:
results, decisions, wl_scores, qc_summary = select_wavelengths(
    data,
    min_corr_coeff=0.99,   # allow slightly worse calibration
    max_conc_rsd=10,       # allow noisier replicates
    max_bias=20,           # allow more CRM deviation
    use_q_flag=False,      # ignore Q flags
    use_rse=True           # still enforce RSE
)
report = diagnostic_report(data, wl_scores, decisions)

In [ ]:
data['NumberBeans'] = data.Description.dropna().apply(lambda x: x.split(', ')[0])
data['ExperimentReplicate'] = data.Description.dropna().apply(lambda x: x.split(', ')[-1])

In [ ]:
data.ExperimentReplicate.dropna()

In [ ]:
data['SampleName'] = data.Label + data.ExperimentReplicate

In [ ]:
data.iloc[0]

In [ ]:
data.Flags.value_counts()

In [ ]:
caldata = data[~data.Concentration.isin(('Uncal','####'))].copy()
caldata.Concentration = caldata.Concentration.astype(float)

In [ ]:
caldata[caldata.Concentration > 0]

In [ ]:
# Selected data for analysis
seldata = caldata[caldata.ExperimentReplicate.isin(('R1','R2','R3'))]

In [ ]:
# Elemental data
eldata = seldata.groupby(['SampleName','Element Label'])['Concentration'].mean().unstack().T
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(eldata)
    #print(eldata[data.Label.drop_duplicates()])

In [ ]:
elmeans = eldata.mean(axis=1)
print(
    '< 0.05:', (elmeans < 0.05).sum(),
    '\n> 0.5:', (elmeans > 0.5).sum(),
    '\n0.05 =< =< 0.5:', ((0.05 < elmeans) & (elmeans < 0.5)).sum(),
)
elmeans

In [ ]:
samplemeans = eldata.mean(axis=0)
samplemeans

In [ ]:
samplemeans_df = pd.DataFrame(samplemeans)
samplemeans_df['replicate'] = [i[-2:] for i in samplemeans_df.index]
samplemeans_df.groupby('replicate').mean()

In [ ]:
samplemeans_df.groupby('replicate').var()